**calc95pct.ipynb**
- Calculates mean daily temperature (tas) climatology for 1979-2000 using AUS-11 (BARRA-R2).

**Reads:** "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/1hr/tas/latest/" \
**Writes:** "ID_HW_BARRA/data/preprocess/t95_baseline.nc" \
**Compute:** xxlarge (28CPU, 126GB) \
**Environment:** analysis3

In [1]:
import xarray as xr, netCDF4 as nc, numpy as np, pandas as pd, os
from pathlib import Path

import dask
import dask.array as da
from dask.distributed import LocalCluster, Client, wait
from datetime import datetime

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')
workingDir = Path().absolute()
print(f"{workingDir}")

/g/data/ng72/ms5578/ID_HW_BARRA


In [3]:
cluster = LocalCluster(
    n_workers=14,
    threads_per_worker=1, 
    memory_limit="9GB"   
)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 14
Total threads: 14,Total memory: 117.35 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:43311,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:37563,Total threads: 1
Dashboard: /proxy/37729/status,Memory: 8.38 GiB
Nanny: tcp://127.0.0.1:34801,


2026-04-13 16:13:19,672 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 243f92b9bc952b6281a0b9bd591f2041 initialized by task ('rechunk-merge-rechunk-transfer-717de225ed5782f33d0813d36a93335e', 1, 2, 0, 2, 3, 61) executed on worker tcp://127.0.0.1:40561
2026-04-13 16:13:19,732 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 753c5a801a43e9f432e3e9b64db13069 initialized by task ('rechunk-merge-rechunk-transfer-717de225ed5782f33d0813d36a93335e', 2, 1, 0, 3, 2, 61) executed on worker tcp://127.0.0.1:40561
2026-04-13 16:13:22,051 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 51e2b00978206d7104d462f69c487096 initialized by task ('rechunk-merge-rechunk-transfer-717de225ed5782f33d0813d36a93335e', 0, 1, 0, 0, 2, 61) executed on worker tcp://127.0.0.1:45915
2026-04-13 16:13:22,052 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 64aa5611f785b422852e7ee66c7a564b initialized by task ('rechunk-merge-rechunk-transfer-717de225ed5782f33d0813d36a93335e', 

In [4]:
tas_path = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/1hr/tas/latest/"
write_path = f'{workingDir}/data/preprocess/'

In [5]:
sdate, edate ='19790101', '20001231'

In [6]:
fdates = [m.strftime('%Y%m') for m in pd.date_range(sdate, edate, freq='ME')]
fnames = [s for s in os.listdir(tas_path) if any(f in s for f in fdates)]
fpaths = sorted([tas_path + f for f in fnames])

# Open hourly tas
tas_ds = xr.open_mfdataset(
    fpaths,
    concat_dim='time',
    combine='nested',
    parallel=True,
    data_vars='minimal',
    coords='minimal',
    drop_variables='time_bnds',
    chunks='auto',
    compat='no_conflicts'
)

In [7]:
tas = tas_ds.tas
tas = tas_ds.tas.chunk({"time": 240, "lat": 323, "lon": 541})
tas

<xarray.DataArray 'tas' (time: 192864, lat: 646, lon: 1082)> Size: 1TB
dask.array<rechunk-merge, shape=(192864, 646, 1082), dtype=float64, chunksize=(240, 323, 541), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 2MB 1979-01-01 ... 2000-12-31T23:00:00
  * lat      (lat) float64 5kB -57.97 -57.86 -57.75 -57.64 ... 12.76 12.87 12.98
  * lon      (lon) float64 9kB 88.48 88.59 88.7 88.81 ... 207.2 207.3 207.4
    height   float64 8B 1.5
    crs      int32 4B 0
Attributes:
    long_name:      Near-Surface Air Temperature
    standard_name:  air_temperature
    units:          K
    cell_methods:   time: point (interval: 1H)
    grid_mapping:   crs

In [8]:
# Derive daily Tmax/Tmin from hourly tas in fixed AEST (UTC+10), then daily mean (tmax+tmin)/2
# shift nominal UTC timestamps to fixed AEST (UTC+10)
time_utc = pd.DatetimeIndex(tas.time.values)
time_aest = time_utc + pd.Timedelta(hours=10)
tas = tas.assign_coords(time=time_aest)

tas_daily_max = tas.resample(time='1D').max()
tas_daily_min = tas.resample(time='1D').min()
tas_daily_mean = (tas_daily_max + tas_daily_min) / 2.0
tas_daily = tas_daily_mean.to_dataset(name='tas')

datestr = f"s{sdate}_e{edate}"

# 95th percentile of daily mean temperature over time
t95 = tas_daily.tas.quantile(0.95, dim='time')
t95 = t95.to_dataset(name='PRCTILE95')

In [9]:
# clear stale encodings
try:
    t95.encoding.clear()
    for v in t95.data_vars:
        t95[v].encoding.clear()
except Exception:
    pass

encoding = {"PRCTILE95":{"zlib": True, "complevel": 4, "shuffle": True}}

In [10]:
t95 = t95.compute() # materialize to memory to avoid 'tas' backend refs
tas_ds.close()

t95.to_netcdf(f'{write_path}t95_baseline.nc',
              encoding=encoding,
              engine='h5netcdf')

/g/data/xp65/public/apps/med_conda/envs/analysis3-26.03/lib/python3.12/site-packages/distributed/client.py:3387: UserWarning: Sending large graph of size 128.97 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
2026-04-13 16:23:40,312 - distributed.worker.memory - WARNING - gc.collect() took 1.024s. This is usually a sign that some tasks handle too many Python objects at the same time. Rechunking the work into smaller tasks might help.
2026-04-13 16:33:07,805 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 5.04 GiB 